# CARELAB · Presencia horaria desde GeoVictoria

Importa las marcas de asistencia de **CARELAB** (API *Customer* de GeoVictoria) y construye
una **matriz binaria horaria de presencia**:

- **Filas** — cada `(día, hora)` del periodo (índice horario). La ventana de horas se
  **deriva de los datos** (mín–máx hora observada en las marcas).
- **Columnas** — los usuarios **con al menos una marca** en el periodo (`Nombre Apellido`).
- **Valores** — `1` si el usuario estuvo **dentro del almacén** en esa hora, `0` si no.

> **Dentro del almacén** = entre el **primer Ingreso** y la **última Salida** del día (la
> colación cuenta como dentro). Una hora vale `1` si ese tramo se solapa con la hora.

**Fuente:** `POST /Login` → `/User/ListComplete` → `/AttendanceBook`. La cuenta (CARELAB)
la fija el token; credenciales en `.env` (`CARELAB_GEOVICTORIA_USER` / `_PASS`).

## 1. Configuración y cliente

In [1]:
import os
import time
import datetime as dt
from collections import Counter

import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()

# ── Credenciales y endpoint ───────────────────────────────────────────────────
BASE_URL = "https://customerapi.geovictoria.com/api/v1"   # entorno productivo
GV_USER = os.getenv("CARELAB_GEOVICTORIA_USER")
GV_PASS = os.getenv("CARELAB_GEOVICTORIA_PASS")
assert GV_USER and GV_PASS, "Faltan CARELAB_GEOVICTORIA_USER / CARELAB_GEOVICTORIA_PASS en .env"

# ── Periodo ───────────────────────────────────────────────────────────────────
DIAS_HISTORIA = 90                       # "últimos 90 días" (parámetro principal)
FECHA_FIN = dt.date.today()
FECHA_INI = FECHA_FIN - dt.timedelta(days=DIAS_HISTORIA)

# ── Alcance de usuarios ───────────────────────────────────────────────────────
SOLO_HABILITADOS = True   # candidatos = Enabled==1; luego se filtran a "con marcas"

# ── Límites de la API CARELAB (hallados por sondeo) ───────────────────────────
#   • registros por llamada (usuarios × días) ≤ ~1400
#   • máximo de días por llamada               → 90 días no caben: se trocea en ventanas
#   • rate limit                               → 3 llamadas / 1 s
VENTANA_DIAS = 14      # días por llamada
CHUNK_USUARIOS = 100   # usuarios por llamada  (100 × 14 = 1400 registros, validado)
PAUSA_SEG = 0.5        # throttle entre llamadas (≈2/seg, bajo el límite de 3/seg)

print(f"Periodo: {FECHA_INI:%d/%m/%Y} → {FECHA_FIN:%d/%m/%Y}  ({DIAS_HISTORIA} días)")

Periodo: 27/03/2026 → 25/06/2026  (90 días)


In [2]:
class GeoVictoriaError(Exception):
    """Error de comunicación con la API de GeoVictoria."""


class GeoVictoriaClient:
    """Cliente mínimo de la *Customer API* de GeoVictoria.

    Solo los endpoints que usa esta notebook: ``/Login``, ``/User/ListComplete`` y
    ``/AttendanceBook``. La cuenta (CARELAB) la determina el token. Reintenta ante el
    rate limit (HTTP 429) y aplica un throttle preventivo entre llamadas.
    """

    def __init__(self, user, password, base_url=BASE_URL, pausa=PAUSA_SEG):
        self.user, self.password = user, password
        self.base_url = base_url.rstrip("/")
        self.pausa = pausa
        self.s = requests.Session()
        self.s.headers.update({"Accept": "application/json",
                               "Content-Type": "application/json"})

    def _post(self, path, payload, timeout=300, reintentos=5):
        url = f"{self.base_url}{path}"
        for intento in range(reintentos):
            time.sleep(self.pausa)                      # throttle preventivo
            r = self.s.post(url, json=payload, timeout=timeout)
            if r.status_code == 429:                    # rate limit → backoff y reintento
                time.sleep(float(r.headers.get("Retry-After", 1.5 * (intento + 1))))
                continue
            if r.status_code != 200:
                raise GeoVictoriaError(f"[HTTP {r.status_code}] {path} → {r.text[:200]}")
            return r.json()
        raise GeoVictoriaError(f"Rate limit persistente en {path} tras {reintentos} intentos")

    def login(self):
        body = self._post("/Login", {"User": self.user, "Password": self.password}, timeout=30)
        token = body.get("token", "")
        if not token:
            raise GeoVictoriaError(f"Login sin token: {str(body)[:200]}")
        self.s.headers["Authorization"] = f"Bearer {token}"
        return token

    def list_users(self):
        body = self._post("/User/ListComplete", {}, timeout=60)
        return body if isinstance(body, list) else body.get("Response", [])

    def attendance_book(self, ini, fin, user_ids):
        """``ini``/``fin``: date; ``user_ids``: lista de Identifier. Devuelve la lista 'Users'."""
        payload = {
            "StartDate": ini.strftime("%Y%m%d") + "000000",
            "EndDate": fin.strftime("%Y%m%d") + "235959",
            "UserIds": ",".join(user_ids),
        }
        body = self._post("/AttendanceBook", payload)
        return body.get("Users", []) if isinstance(body, dict) else (body or [])

## 2. Autenticación y catálogo de usuarios

In [3]:
gv = GeoVictoriaClient(GV_USER, GV_PASS)
token = gv.login()
print(f"✓ Autenticado en CARELAB — token JWT recibido (len={len(token)})")

✓ Autenticado en CARELAB — token JWT recibido (len=383)


In [4]:
usuarios = pd.DataFrame(gv.list_users())
for col in ("Identifier", "Name", "LastName", "Enabled"):
    if col not in usuarios.columns:
        usuarios[col] = pd.NA

candidatos = usuarios.copy()
if SOLO_HABILITADOS:
    candidatos = candidatos[candidatos["Enabled"].astype(str) == "1"]
ids_candidatos = candidatos.loc[candidatos["Identifier"].notna(), "Identifier"].astype(str).tolist()

print(f"Usuarios en CARELAB: {len(usuarios)}  |  "
      f"candidatos ({'habilitados' if SOLO_HABILITADOS else 'todos'}): {len(ids_candidatos)}")
usuarios[["Identifier", "Name", "LastName", "Enabled"]].head()

Usuarios en CARELAB: 2651  |  candidatos (habilitados): 1206


,Identifier,Name,LastName,Enabled
0,01048510083,Liliana,Rubio arias,0
1,01169871355,Ricardo bryan,Figueroa venegas,0
2,02219464753,Tania itzel,Santillan muñoz,0
3,04139855482,Marcos eduardo,Hernandez hernandez,0
4,04169705086,Olga nallely,Miranda jeronimo,0


## 3. Descarga de marcas (Ingreso / Salida)

Se descarga el libro de asistencia troceado en **ventanas de ≤14 días × lotes de ≤100
usuarios** (= 1400 registros/llamada, el tope de la API) con *throttle* y reintento ante el
*rate limit* (3 llamadas/seg). De cada usuario se extraen recursivamente sus marcas
`Ingreso`/`Salida` (el dict de marca no trae el identificador del usuario).

In [5]:
def _chunks(seq, n):
    for i in range(0, len(seq), n):
        yield seq[i:i + n]


def _ventanas(ini, fin, dias):
    cur = ini
    while cur <= fin:
        w_fin = min(cur + dt.timedelta(days=dias - 1), fin)
        yield cur, w_fin
        cur = w_fin + dt.timedelta(days=1)


def extraer_marcas(obj, identifier, acc):
    """Recorre el subárbol de un usuario y junta sus marcas Ingreso/Salida.

    El dict de marca (``PunchItem``) no incluye el identificador del usuario, por eso se
    etiqueta con el ``identifier`` del usuario que se está iterando.
    """
    if isinstance(obj, dict):
        d = str(obj.get("Date", ""))
        if d.isdigit() and len(d) == 14 and obj.get("Type") in ("Ingreso", "Salida"):
            acc.append({"Identifier": identifier, "PunchId": obj.get("PunchId"),
                        "ts": d, "Type": obj["Type"]})
        for v in obj.values():
            extraer_marcas(v, identifier, acc)
    elif isinstance(obj, list):
        for v in obj:
            extraer_marcas(v, identifier, acc)


ventanas = list(_ventanas(FECHA_INI, FECHA_FIN, VENTANA_DIAS))
lotes = list(_chunks(ids_candidatos, CHUNK_USUARIOS))
total = len(ventanas) * len(lotes)
print(f"Descargando marcas: {len(lotes)} lotes × {len(ventanas)} ventanas = {total} llamadas")

_marcas, hechas = [], 0
for w_ini, w_fin in ventanas:
    antes = len(_marcas)
    for lote in lotes:
        for u in gv.attendance_book(w_ini, w_fin, lote):
            extraer_marcas(u, str(u.get("Identifier")), _marcas)
        hechas += 1
    print(f"  ventana {w_ini:%d/%m/%y}–{w_fin:%d/%m/%y}: +{len(_marcas) - antes:>6} marcas "
          f"({hechas}/{total} llamadas)")

marcas = pd.DataFrame(_marcas)
if not marcas.empty:
    marcas["ts"] = pd.to_datetime(marcas["ts"], format="%Y%m%d%H%M%S")
    clave = ["Identifier", "PunchId"] if marcas["PunchId"].notna().any() else ["Identifier", "ts", "Type"]
    marcas = marcas.drop_duplicates(subset=clave).reset_index(drop=True)

n_users = marcas["Identifier"].nunique() if not marcas.empty else 0
print(f"\nMarcas Ingreso/Salida: {len(marcas)}  |  usuarios con marcas: {n_users}")
marcas.head()

Descargando marcas: 13 lotes × 7 ventanas = 91 llamadas


  ventana 27/03/26–09/04/26: + 17424 marcas (13/91 llamadas)


  ventana 10/04/26–23/04/26: + 18688 marcas (26/91 llamadas)


  ventana 24/04/26–07/05/26: + 17062 marcas (39/91 llamadas)


  ventana 08/05/26–21/05/26: + 18612 marcas (52/91 llamadas)


  ventana 22/05/26–04/06/26: + 17855 marcas (65/91 llamadas)


  ventana 05/06/26–18/06/26: + 16305 marcas (78/91 llamadas)


  ventana 19/06/26–25/06/26: +  7391 marcas (91/91 llamadas)



Marcas Ingreso/Salida: 113337  |  usuarios con marcas: 761


,Identifier,PunchId,ts,Type
0,94057600531,5v8IAn_HOqb5M3kCVS6GQg,2026-04-03 12:00:00,Ingreso
1,94057600531,XAiFDw2N0o7wNzLb9MHOxw,2026-04-03 16:00:00,Salida
2,94057600531,UrrXNlXTyK4Wni2jrQ5qwA,2026-04-03 16:37:00,Ingreso
3,94057600531,fSlCZoIDkaRTWtpLjbSzig,2026-04-03 18:01:00,Salida
4,94057600531,BVaIpZA-5Mz2iQrp3-HQ2w,2026-04-05 07:03:00,Ingreso


## 4. Tramos de presencia por usuario-día  (1ª entrada → última salida)

In [6]:
# Si falta un lado (olvido de marca), se usa la primera/última marca del día.
if marcas.empty:
    tramos = pd.DataFrame(columns=["Identifier", "fecha", "inicio", "fin"])
else:
    marcas["fecha"] = marcas["ts"].dt.normalize()
    ing = (marcas[marcas["Type"] == "Ingreso"]
           .groupby(["Identifier", "fecha"])["ts"].min().rename("inicio"))
    sal = (marcas[marcas["Type"] == "Salida"]
           .groupby(["Identifier", "fecha"])["ts"].max().rename("fin"))
    mn = marcas.groupby(["Identifier", "fecha"])["ts"].min().rename("min_ts")
    mx = marcas.groupby(["Identifier", "fecha"])["ts"].max().rename("max_ts")
    tramos = pd.concat([ing, sal, mn, mx], axis=1).reset_index()
    tramos["inicio"] = tramos["inicio"].fillna(tramos["min_ts"])
    tramos["fin"] = tramos["fin"].fillna(tramos["max_ts"])
    invertido = tramos["fin"] < tramos["inicio"]
    tramos.loc[invertido, "fin"] = tramos.loc[invertido, "inicio"]
    tramos = tramos[["Identifier", "fecha", "inicio", "fin"]]

print(f"Tramos (usuario-día con presencia): {len(tramos)}")
tramos.head()

Tramos (usuario-día con presencia): 45574


,Identifier,fecha,inicio,fin
0,01038421119,2026-03-27,2026-03-27 07:46:00,2026-03-27 18:04:00
1,01038421119,2026-04-01,2026-04-01 07:48:00,2026-04-01 18:03:00
2,01038421119,2026-04-02,2026-04-02 07:42:00,2026-04-02 16:03:00
3,01038421119,2026-04-07,2026-04-07 09:03:00,2026-04-07 09:03:00
4,01038421119,2026-04-08,2026-04-08 07:46:00,2026-04-08 18:01:00


## 5. Matriz binaria horaria

In [7]:
def _s(x):
    return "" if pd.isna(x) else str(x)


# ── Ventana horaria derivada de los datos ─────────────────────────────────────
if tramos.empty:
    HORA_MIN, HORA_MAX = 0, 23
else:
    HORA_MIN = int(tramos["inicio"].dt.hour.min())
    HORA_MAX = int(tramos["fin"].dt.hour.max())
HORAS = list(range(HORA_MIN, HORA_MAX + 1))

# ── Índice de filas: cada (día, hora) del periodo dentro de la ventana ────────
dias = pd.date_range(FECHA_INI, FECHA_FIN, freq="D")
indice = pd.DatetimeIndex([d + pd.Timedelta(hours=h) for d in dias for h in HORAS], name="hora")

# ── Columnas: usuarios con marcas, etiquetados "Nombre Apellido" ──────────────
info = usuarios.copy()
info["Identifier"] = info["Identifier"].astype(str)
info = info.drop_duplicates("Identifier").set_index("Identifier")

etiqueta = {}
for uid in (tramos["Identifier"].astype(str).unique() if not tramos.empty else []):
    if uid in info.index:
        nom = (_s(info.loc[uid, "Name"]) + " " + _s(info.loc[uid, "LastName"])).strip()
    else:
        nom = ""
    etiqueta[uid] = nom or uid

# desambigua nombres repetidos añadiendo el Identifier
repetidos = Counter(etiqueta.values())
for uid, lab in list(etiqueta.items()):
    if repetidos[lab] > 1:
        etiqueta[uid] = f"{lab} ({uid})"
columnas = sorted(set(etiqueta.values()))

# ── Relleno de la matriz (1 si el tramo se solapa con la hora) ────────────────
row_pos = {ts: i for i, ts in enumerate(indice)}
col_pos = {c: j for j, c in enumerate(columnas)}
M = np.zeros((len(indice), len(columnas)), dtype=np.int8)
for r in tramos.itertuples(index=False):
    lab = etiqueta.get(str(r.Identifier))
    if lab is None:
        continue
    j = col_pos[lab]
    h0 = r.inicio.floor("h")
    h1 = (r.fin - pd.Timedelta(seconds=1)).floor("h")
    if h1 < h0:
        h1 = h0
    for ts in pd.date_range(h0, h1, freq="h"):
        i = row_pos.get(ts)
        if i is not None:
            M[i, j] = 1

df = pd.DataFrame(M, index=indice, columns=columnas)
print(f"Ventana horaria (derivada de los datos): {HORA_MIN:02d}:00–{HORA_MAX:02d}:00  "
      f"({len(HORAS)} horas/día)")
print(f"Matriz: {df.shape[0]} filas (hora×día) × {df.shape[1]} usuarios")

Ventana horaria (derivada de los datos): 00:00–23:00  (24 horas/día)
Matriz: 2184 filas (hora×día) × 761 usuarios


## 6. Resumen

In [8]:
activos = int(df.values.sum())
print(f"Periodo               : {FECHA_INI:%d/%m/%Y} → {FECHA_FIN:%d/%m/%Y}")
print(f"Ventana horaria       : {HORA_MIN:02d}:00–{HORA_MAX:02d}:00 ({len(HORAS)} h/día)")
print(f"Filas (hora × día)    : {df.shape[0]:,}")
print(f"Usuarios (columnas)   : {df.shape[1]:,}")
print(f"Celdas '1' (h-persona): {activos:,}  ({activos / df.size:.1%} de la matriz)")

Periodo               : 27/03/2026 → 25/06/2026
Ventana horaria       : 00:00–23:00 (24 h/día)
Filas (hora × día)    : 2,184
Usuarios (columnas)   : 761
Celdas '1' (h-persona): 425,406  (25.6% de la matriz)


## 7. Vista del dataset (`df.head()`)

In [9]:
df.head()

,ABAD GARZON PAULINO,ABRAHAM ISLAS MARQUEZ,ABRIL SILVA ADAME,ADBEEL ZURIZADAY FERNANDEZ RODRIGUEZ,ADRIANA ATZEL CORNEJO CRUZ,ADRIANA ESCAMILLA ORTEGA,ADRIANA GONZALEZ GUTIERREZ,ADRIANA GUADALUPE MUJICA MANZO,ADRIANA MARTINEZ PERALTA,AILYN DANIELA NAVA SANTOS,...,Yesica Ruiz alanis,Yesmin valentina Tavera tinoco,Yessica Gomez padilla,Yessica yadira Arreola rodriguez,Yolanda Castañeda Enriquez,Yuliana Arreola amezcua,Yuriana Campos ortiz,Yuridia Mendoza Arellano,ZABDIEL SILOE LOPEZ GARCIA,Zayra milagros Zuluaga lopez
hora,,,,,,,,,,,,,,,,,,,,,
2026-03-27 00:00:00,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2026-03-27 01:00:00,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2026-03-27 02:00:00,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2026-03-27 03:00:00,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2026-03-27 04:00:00,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
